<a href="https://colab.research.google.com/github/ypuete10-cmd/ARAP/blob/main/Aip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
from typing import Tuple, List, Optional
from enum import Enum
from dataclasses import dataclass

# =============================================================================
# GAME CONSTANTS
# =============================================================================

class Action(Enum):
    ADVANCE = "advance"
    RETREAT = "retreat"
    PUSH = "push"

class CurrentOutcome(Enum):
    """River current: (probability_weight, movement)"""
    CALM = (2, 0)      # 2/6 chance, no movement
    FLOW_1 = (2, 1)    # 2/6 chance, +1 toward B
    SURGE = (1, 2)     # 1/6 chance, +2 toward B
    EDDY = (1, -1)     # 1/6 chance, -1 toward A

    def __init__(self, weight, movement):
        self.prob = weight / 6
        self.movement = movement

# Probability distribution
CURRENT_PROBS = [(outcome, outcome.prob) for outcome in CurrentOutcome]

MAX_POSITION = 8
MIN_POSITION = 0
MAX_ROUNDS = 100

# =============================================================================
# GAME STATE
# =============================================================================

@dataclass(frozen=True)
class GameState:
    player_pos: int      # 0=Bank A, 8=Bank B
    opponent_pos: int    # 0=Bank A, 8=Bank B
    is_player_turn: bool # True=Player(Max), False=Opponent(Min)
    round_num: int = 0

    def copy(self, **kwargs):
        return GameState(
            kwargs.get('player_pos', self.player_pos),
            kwargs.get('opponent_pos', self.opponent_pos),
            kwargs.get('is_player_turn', self.is_player_turn),
            kwargs.get('round_num', self.round_num)
        )

# =============================================================================
# GAME LOGIC
# =============================================================================

class RiverCrossingGame:
    @staticmethod
    def get_legal_actions(state: GameState) -> List[Action]:
        actions = []
        pos = state.player_pos if state.is_player_turn else state.opponent_pos
        opp_pos = state.opponent_pos if state.is_player_turn else state.player_pos

        # Movement actions (direction depends on player)
        if state.is_player_turn:  # Player moves toward 8
            if pos < MAX_POSITION:
                actions.append(Action.ADVANCE)
            if pos > MIN_POSITION:
                actions.append(Action.RETREAT)
        else:  # Opponent moves toward 0
            if pos > MIN_POSITION:
                actions.append(Action.ADVANCE)
            if pos < MAX_POSITION:
                actions.append(Action.RETREAT)

        # Push if on same stone (not at banks)
        if pos == opp_pos and pos not in (MIN_POSITION, MAX_POSITION):
            actions.append(Action.PUSH)

        return actions

    @staticmethod
    def apply_action(state: GameState, action: Action) -> GameState:
        if state.is_player_turn:
            pos, opp_pos = state.player_pos, state.opponent_pos
            if action == Action.ADVANCE:
                return state.copy(player_pos=min(pos + 1, MAX_POSITION))
            elif action == Action.RETREAT:
                return state.copy(player_pos=max(pos - 1, MIN_POSITION))
            elif action == Action.PUSH:
                return state.copy(opponent_pos=min(opp_pos + 1, MAX_POSITION))
        else:
            pos, opp_pos = state.opponent_pos, state.player_pos
            if action == Action.ADVANCE:
                return state.copy(opponent_pos=max(pos - 1, MIN_POSITION))
            elif action == Action.RETREAT:
                return state.copy(opponent_pos=min(pos + 1, MAX_POSITION))
            elif action == Action.PUSH:
                return state.copy(player_pos=max(opp_pos - 1, MIN_POSITION))
        return state

    @staticmethod
    def apply_current(state: GameState, outcome: CurrentOutcome) -> GameState:
        movement = outcome.movement
        return state.copy(
            player_pos=max(MIN_POSITION, min(MAX_POSITION, state.player_pos + movement)),
            opponent_pos=max(MIN_POSITION, min(MAX_POSITION, state.opponent_pos + movement)),
            is_player_turn=not state.is_player_turn,
            round_num=state.round_num + (0 if state.is_player_turn else 1)
        )

    @staticmethod
    def is_terminal(state: GameState) -> bool:
        return (state.player_pos == MAX_POSITION or
                state.opponent_pos == MIN_POSITION or
                state.round_num >= MAX_ROUNDS)

    @staticmethod
    def get_reward(state: GameState) -> float:
        if not RiverCrossingGame.is_terminal(state):
            return 0.0
        p_win = state.player_pos == MAX_POSITION
        o_win = state.opponent_pos == MIN_POSITION
        if p_win and not o_win:
            return 1.0
        elif o_win and not p_win:
            return -1.0
        return 0.0  # Draw

# =============================================================================
# EXPECTIMINIMAX ALGORITHM
# =============================================================================

class Expectiminimax:
    def __init__(self, max_depth: int = 4):
        self.max_depth = max_depth
        self.nodes_explored = 0
        self.transposition_table = {}

    def search(self, state: GameState, depth: int = 0,
               alpha: float = -float('inf'), beta: float = float('inf')) -> Tuple[float, Optional[Action]]:
        self.nodes_explored += 1

        # Check cache
        key = (state, depth)
        if key in self.transposition_table:
            return self.transposition_table[key]

        # Terminal or depth limit
        if RiverCrossingGame.is_terminal(state):
            return RiverCrossingGame.get_reward(state), None
        if depth >= self.max_depth:
            # Heuristic: progress difference
            val = (state.player_pos - (MAX_POSITION - state.opponent_pos)) / MAX_POSITION * 0.9
            return val, None

        actions = RiverCrossingGame.get_legal_actions(state)
        if not actions:
            return RiverCrossingGame.get_reward(state), None

        best_action = actions[0]

        if state.is_player_turn:  # Max node
            value = -float('inf')
            for action in actions:
                after = RiverCrossingGame.apply_action(state, action)
                exp_val = self._expect(after, depth + 1, alpha, beta)
                if exp_val > value:
                    value, best_action = exp_val, action
                alpha = max(alpha, value)
                if beta <= alpha:
                    break
        else:  # Min node
            value = float('inf')
            for action in actions:
                after = RiverCrossingGame.apply_action(state, action)
                exp_val = self._expect(after, depth + 1, alpha, beta)
                if exp_val < value:
                    value, best_action = exp_val, action
                beta = min(beta, value)
                if beta <= alpha:
                    break

        result = (value, best_action)
        self.transposition_table[key] = result
        return result

    def _expect(self, state: GameState, depth: int, alpha: float, beta: float) -> float:
        """Chance node: expected value over current outcomes"""
        return sum(
            prob * self.search(RiverCrossingGame.apply_current(state, outcome),
                              depth + 1, alpha, beta)[0]
            for outcome, prob in CURRENT_PROBS
        )

    def reset_stats(self):
        self.nodes_explored = 0
        self.transposition_table = {}

# =============================================================================
# AGENTS
# =============================================================================

class RandomAgent:
    def select_action(self, state: GameState) -> Action:
        return random.choice(RiverCrossingGame.get_legal_actions(state))

    def get_name(self) -> str:
        return "Random"

class MinimaxAgent:
    def __init__(self, max_depth: int = 4, verbose: bool = False):
        self.max_depth = max_depth
        self.verbose = verbose
        self.searcher = Expectiminimax(max_depth)

    def select_action(self, state: GameState) -> Action:
        self.searcher.reset_stats()
        value, action = self.searcher.search(state, depth=0)

        if self.verbose:
            print(f"  [Minimax] Value: {value:.3f}, Nodes: {self.searcher.nodes_explored}")

        return action or random.choice(RiverCrossingGame.get_legal_actions(state))

    def get_name(self) -> str:
        return f"Minimax(d={self.max_depth})"

# =============================================================================
# GAME RUNNER
# =============================================================================

class GameRunner:
    def __init__(self, player_agent, opponent_agent, verbose: bool = True):
        self.player_agent = player_agent
        self.opponent_agent = opponent_agent
        self.verbose = verbose

    def play_game(self, seed: Optional[int] = None) -> GameState:
        if seed is not None:
            random.seed(seed)

        state = GameState(0, 8, True, 0)

        if self.verbose:
            print(f"\n{'='*60}")
            print(f"GAME: {self.player_agent.get_name()} vs {self.opponent_agent.get_name()}")
            self._display_state(state)

        while not RiverCrossingGame.is_terminal(state):
            # Get action
            agent = self.player_agent if state.is_player_turn else self.opponent_agent
            action = agent.select_action(state)

            # Apply action
            after = RiverCrossingGame.apply_action(state, action)

            # Roll die for current
            roll = random.randint(1, 6)
            outcome = (
                CurrentOutcome.CALM if roll <= 2 else
                CurrentOutcome.FLOW_1 if roll <= 4 else
                CurrentOutcome.SURGE if roll == 5 else
                CurrentOutcome.EDDY
            )
            state = RiverCrossingGame.apply_current(after, outcome)

            if self.verbose:
                print(f"\n{agent.get_name()}: {action.value} | Die {roll}: {outcome.name}")
                self._display_state(state)

        if self.verbose:
            winner = "Player" if RiverCrossingGame.get_reward(state) > 0 else \
                     "Opponent" if RiverCrossingGame.get_reward(state) < 0 else "Draw"
            print(f"\nRESULT: {winner}")

        return state

    def _display_state(self, state: GameState):
        board = ""
        for i in range(9):
            if state.player_pos == i and state.opponent_pos == i:
                board += "[X]"
            elif state.player_pos == i:
                board += "[P]"
            elif state.opponent_pos == i:
                board += "[O]"
            elif i == 0:
                board += " A "
            elif i == 8:
                board += " B "
            else:
                board += f"[{i}]"
            if i < 8:
                board += "-"
        print(f"Round {state.round_num}: {board}")

# =============================================================================
# HUMAN VS AI
# =============================================================================

class HumanVsAI:
    def __init__(self, ai_depth: int = 4):
        self.ai = MinimaxAgent(max_depth=ai_depth)

    def play(self):
        print("\n" + "="*60)
        print("RIVER CROSSING - YOU vs AI")
        print("You: Bank A (0) → Bank B (8) | AI: Bank B (8) → Bank A (0)")
        print("="*60)

        state = GameState(0, 8, True, 0)

        while not RiverCrossingGame.is_terminal(state):
            self._show_board(state)

            if state.is_player_turn:
                action = self._get_human_move(state)
            else:
                print("\nAI thinking...")
                action = self.ai.select_action(state)
                print(f"AI plays: {action.value}")

            after = RiverCrossingGame.apply_action(state, action)

            # Roll die
            roll = random.randint(1, 6)
            outcome = (
                CurrentOutcome.CALM if roll <= 2 else
                CurrentOutcome.FLOW_1 if roll <= 4 else
                CurrentOutcome.SURGE if roll == 5 else
                CurrentOutcome.EDDY
            )
            state = RiverCrossingGame.apply_current(after, outcome)

            print(f" Roll {roll}: {outcome.name} ({outcome.movement:+d})")
            if outcome.movement != 0:
                print(f"   You: {after.player_pos} → {state.player_pos}")
                print(f"   AI:  {after.opponent_pos} → {state.opponent_pos}")

            if not RiverCrossingGame.is_terminal(state):
                input("\nPress Enter to continue...")

        self._show_board(state)
        reward = RiverCrossingGame.get_reward(state)
        print("\n" + "="*60)
        print(" YOU WIN!" if reward > 0 else " AI WINS!" if reward < 0 else " DRAW!")
        print("="*60)

    def _show_board(self, state: GameState):
        print("\n" + "-"*40)
        print(f"ROUND {state.round_num} | {'YOUR TURN' if state.is_player_turn else 'AI TURN'}")
        print(f"You (P): {state.player_pos}/8 | AI (O): {8-state.opponent_pos}/8")

        board = ""
        for i in range(9):
            if state.player_pos == i and state.opponent_pos == i:
                board += "[X]"
            elif state.player_pos == i:
                board += "[P]"
            elif state.opponent_pos == i:
                board += "[O]"
            elif i == 0:
                board += " A "
            elif i == 8:
                board += " B "
            else:
                board += f"[{i}]"
            if i < 8:
                board += "-"
        print(board)
        print("Current: [1-2]Calm [3-4]+1B [5]+2B [6]-1A")

    def _get_human_move(self, state: GameState) -> Action:
        actions = RiverCrossingGame.get_legal_actions(state)

        print("\nYour options:")
        for i, act in enumerate(actions, 1):
            desc = {"advance": "Move toward B", "retreat": "Move toward A", "push": "Push opponent"}
            print(f"  {i}. {act.value.upper():<8} - {desc[act.value]}")

        while True:
            choice = input("\nChoose (1-{} or name): ".format(len(actions))).strip().lower()

            # Try number
            try:
                idx = int(choice) - 1
                if 0 <= idx < len(actions):
                    return actions[idx]
            except ValueError:
                pass

            # Try name
            for act in actions:
                if choice == act.value:
                    return act

            print("Invalid choice. Try again.")

# =============================================================================
# EXPERIMENT & EXPLANATION
# =============================================================================

class Experiment:
    def __init__(self, num_games: int = 100):
        self.num_games = num_games

    def run(self, player, opponent, verbose: bool = False):
        wins = losses = draws = 0
        total_rounds = 0

        for i in range(self.num_games):
            runner = GameRunner(player, opponent, verbose=False)
            final = runner.play_game(seed=i)

            reward = RiverCrossingGame.get_reward(final)
            total_rounds += final.round_num

            if reward > 0:
                wins += 1
            elif reward < 0:
                losses += 1
            else:
                draws += 1

            if verbose and (i + 1) % 10 == 0:
                print(f"  Progress: {i+1}/{self.num_games} (W:{wins} L:{losses} D:{draws})")

        return {
            'wins': wins, 'losses': losses, 'draws': draws,
            'win_rate': wins / self.num_games,
            'avg_rounds': total_rounds / self.num_games
        }

    def print_stats(self, stats: dict, name: str):
        print(f"\n{'='*60}")
        print(f"RESULTS: {name}")
        print(f"Wins: {stats['wins']} ({stats['win_rate']*100:.1f}%) | "
              f"Losses: {stats['losses']} | Draws: {stats['draws']}")
        print(f"Avg rounds: {stats['avg_rounds']:.1f}")

class DecisionExplainer:
    def explain(self, state: GameState, depth: int = 3):
        print(f"\n{'='*60}")
        print("DECISION EXPLANATION")
        print(f"State: P={state.player_pos}, O={state.opponent_pos} | "
              f"Turn: {'Player' if state.is_player_turn else 'Opponent'}")

        actions = RiverCrossingGame.get_legal_actions(state)
        searcher = Expectiminimax(max_depth=depth)

        print(f"\n{'Action':<10} {'Expected Value':<15} Details")
        print("-" * 50)

        for action in actions:
            after = RiverCrossingGame.apply_action(state, action)
            expected = 0
            details = []

            for outcome, prob in CURRENT_PROBS:
                new_state = RiverCrossingGame.apply_current(after, outcome)
                val, _ = searcher.search(new_state, depth=1)
                expected += prob * val
                details.append(f"{outcome.name[:4]}:{val:+.2f}")

            print(f"{action.value:<10} {expected:>+7.3f}        {', '.join(details)}")

        searcher.reset_stats()
        best_val, best_action = searcher.search(state, depth=0)
        print(f"\nBEST: {best_action.value.upper() if best_action else 'None'} "
              f"(value: {best_val:+.3f}, nodes: {searcher.nodes_explored})")

# =============================================================================
# MAIN
# =============================================================================

def show_instructions():
    print("\n" + "="*60)
    print("HOW TO PLAY")
    print("="*60)
    print("OBJECTIVE: You (P) start at Bank A (0), reach Bank B (8)")
    print("           AI (O) starts at Bank B (8), reaches Bank A (0)")
    print("\nACTIONS:")
    print("  ADVANCE - Move toward your goal")
    print("  RETREAT - Move back (sometimes strategic)")
    print("  PUSH    - Push opponent back (only on same stone)")
    print("\nRIVER CURRENT (die roll after each move):")
    print("  [1-2] CALM  - No change")
    print("  [3-4] FLOW  - Both +1 toward B (helps you!)")
    print("  [5]   SURGE - Both +2 toward B (big help!)")
    print("  [6]   EDDY  - Both -1 toward A (danger!)")
    print("="*60)

def main():
    print("="*60)
    print("RIVER CROSSING")
    print("="*60)

    # Instructions
    show_instructions()

    # Play vs AI?
    try:
        if input("\nPlay against AI? (y/n): ").strip().lower() in ('y', 'yes'):
            HumanVsAI(ai_depth=4).play()
    except (EOFError, KeyboardInterrupt):
        pass

    # Experiments
    print("\n" + "="*60)
    print("100-GAME EXPERIMENTS")
    print("="*60)

    exp = Experiment(100)

    # Random vs Minimax
    stats1 = exp.run(RandomAgent(), MinimaxAgent(4), verbose=True)
    exp.print_stats(stats1, "Random vs Minimax (Random plays first)")

    # Minimax vs Random
    stats2 = exp.run(MinimaxAgent(4), RandomAgent(), verbose=True)
    exp.print_stats(stats2, "Minimax vs Random (Minimax plays first)")

    # Summary
    print("\n" + "="*60)
    print("SUMMARY")
    print(f"{'Matchup':<30} {'Win%':>8}")
    print("-" * 40)
    print(f"{'Random vs Minimax':<30} {stats1['win_rate']*100:>7.1f}%")
    print(f"{'Minimax vs Random':<30} {stats2['win_rate']*100:>7.1f}%")
    print("="*60)

    # Decision explanation
    print("\n" + "="*60)
    print("AI DECISION EXAMPLE")
    print("="*60)
    DecisionExplainer().explain(GameState(3, 5, True, 2), depth=3)

if __name__ == "__main__":
    main()

RIVER CROSSING

HOW TO PLAY
OBJECTIVE: You (P) start at Bank A (0), reach Bank B (8)
           AI (O) starts at Bank B (8), reaches Bank A (0)

ACTIONS:
  ADVANCE - Move toward your goal
  RETREAT - Move back (sometimes strategic)
  PUSH    - Push opponent back (only on same stone)

RIVER CURRENT (die roll after each move):
  [1-2] CALM  - No change
  [3-4] FLOW  - Both +1 toward B (helps you!)
  [5]   SURGE - Both +2 toward B (big help!)
  [6]   EDDY  - Both -1 toward A (danger!)

RIVER CROSSING - YOU vs AI
You: Bank A (0) → Bank B (8) | AI: Bank B (8) → Bank A (0)

----------------------------------------
ROUND 0 | YOUR TURN
You (P): 0/8 | AI (O): 0/8
[P]-[1]-[2]-[3]-[4]-[5]-[6]-[7]-[O]
Current: [1-2]Calm [3-4]+1B [5]+2B [6]-1A

Your options:
  1. ADVANCE  - Move toward B
 Roll 3: FLOW_1 (+1)
   You: 1 → 2
   AI:  8 → 8

----------------------------------------
ROUND 0 | AI TURN
You (P): 2/8 | AI (O): 0/8
 A -[1]-[P]-[3]-[4]-[5]-[6]-[7]-[O]
Current: [1-2]Calm [3-4]+1B [5]+2B [6]-1A
